<a href="https://colab.research.google.com/github/shanmugasree312/Industrial-Asset-Health-Guardian/blob/main/Asset_Health_Guardian_DigitalTwin_ipynb_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import hashlib
import requests
import json
from datetime import datetime
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

# --- CONFIGURATION ---
# This is your correct Firebase Database API URL
FIREBASE_URL = "https://assetguardian-1cde2-default-rtdb.firebaseio.com/"

# 1. Initialize State & Ledger
ledger = [{"Timestamp": "Genesis Block", "Alert": "System Initialized", "Hash": "0"*64}]
machine_health = 100.0  # Starts at 100% health

def create_hash(data, previous_hash):
    return hashlib.sha256(f"{data}{previous_hash}".encode()).hexdigest()

def push_to_cloud(data_payload):
    # Automatically formats the URL correctly
    base_url = FIREBASE_URL.strip('/')
    full_url = f"{base_url}/security_ledger.json"

    try:
        response = requests.post(full_url, json=data_payload)
        if response.status_code == 200:
            print(f"\n✅ SUCCESS! Cloud Sync Complete. Firebase generated ID: {response.json().get('name')}")
        else:
            print(f"\n❌ FIREBASE REJECTED IT! Error {response.status_code}: {response.text}")
    except Exception as e:
        print(f"\n❌ NETWORK ERROR: {e}")

# 2. Train the Edge AI Model
normal_baseline = np.random.normal(loc=[50, 60, 5], scale=[5, 2, 0.5], size=(300, 3))
ai_model = IsolationForest(contamination=0.05, random_state=42)
ai_model.fit(normal_baseline)

# 3. Create DataFrame for Telemetry & RUL
telemetry_history = pd.DataFrame(columns=["Time", "Vibration_Hz", "Temp_C", "Current_A", "Status", "Health_Pct", "RUL_mins"])

print("Status: AI Trained, Genesis Block Created, Cloud Sync Ready.")

Status: AI Trained, Genesis Block Created, Cloud Sync Ready.


In [19]:
import time
from IPython.display import clear_output, display
import ipywidgets as widgets
import matplotlib.pyplot as plt

def get_sensor_reading(anomaly=False):
    if anomaly:
        return [np.random.normal(125, 8), np.random.normal(88, 3), np.random.normal(16, 1)]
    return [np.random.normal(50, 4), np.random.normal(60, 2), np.random.normal(5, 0.4)]

def run_simulation(inject_anomaly=False):
    global telemetry_history, ledger, machine_health

    for i in range(8):
        is_spike = inject_anomaly and (i == 4)
        reading = get_sensor_reading(anomaly=is_spike)
        t_stamp = datetime.now().strftime("%H:%M:%S")

        # 1. AI Inference
        is_anomaly = ai_model.predict([reading])[0] == -1
        status_label = "CRITICAL ANOMALY" if is_anomaly else "Normal"

        # 2. Predictive Maintenance (Remaining Useful Life Math)
        # Normal wear is 0.2% per cycle. Anomalies cause severe 8% degradation.
        degradation_rate = 8.0 if is_anomaly else 0.2
        machine_health = max(0, machine_health - degradation_rate)

        # Calculate Remaining Useful Life (Assuming 1 cycle = 1 minute of operation)
        rul_minutes = machine_health / degradation_rate if degradation_rate > 0 else 999

        # 3. Cryptographic Logging & Cloud Sync
        if is_anomaly:
            alert_msg = f" Asset Failure: Vib={reading[0]:.1f}Hz, Temp={reading[1]:.1f}C"
            block_hash = create_hash(alert_msg, ledger[-1]["Hash"])

            log_entry = {"Timestamp": t_stamp, "Alert": alert_msg, "Hash": block_hash, "Health_Remaining": f"{machine_health:.1f}%"}
            ledger.append(log_entry)

            # Push securely to Firebase
            push_to_cloud(log_entry)

        # Update History
        new_row = pd.DataFrame([{
            "Time": t_stamp, "Vibration_Hz": reading[0], "Temp_C": reading[1], "Current_A": reading[2],
            "Status": status_label, "Health_Pct": machine_health, "RUL_mins": rul_minutes
        }])
        telemetry_history = pd.concat([telemetry_history, new_row], ignore_index=True).tail(25)

    # --- Render UI ---
    with output_area:
        clear_output(wait=True)
        fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 8))

        # Graph 1: Sensor Telemetry
        ax1.plot(telemetry_history["Time"], telemetry_history["Vibration_Hz"], label="Vibration", color="royalblue", marker="o")
        ax1.plot(telemetry_history["Time"], telemetry_history["Temp_C"], label="Temp", color="darkorange", marker="s")
        ax1.set_title("Edge AI Telemetry Stream", fontweight="bold")
        ax1.tick_params(axis='x', rotation=45)
        ax1.grid(True, linestyle="--", alpha=0.6)
        ax1.legend(loc="upper left")

        # Graph 2: Remaining Useful Life (RUL) Curve
        ax2.plot(telemetry_history["Time"], telemetry_history["Health_Pct"], label="Asset Health %", color="purple", linewidth=3)
        ax2.axhline(y=20, color='red', linestyle='--', label="Critical Maintenance Threshold")
        ax2.set_ylim(0, 105)
        ax2.set_title("Predictive Maintenance: Asset Health Degradation", fontweight="bold")
        ax2.tick_params(axis='x', rotation=45)
        ax2.grid(True, linestyle="--", alpha=0.3)
        ax2.legend(loc="lower left")

        # Graph 3: AI Anomaly Flags
        colors = ['red' if s == "CRITICAL ANOMALY" else 'lightgray' for s in telemetry_history["Status"]]
        ax3.bar(telemetry_history["Time"], [1 if s == "CRITICAL ANOMALY" else 0 for s in telemetry_history["Status"]], color=colors)
        ax3.set_yticks([0, 1])
        ax3.set_yticklabels(['Normal', 'Anomaly'])
        ax3.set_title("Isolation Forest Detection", fontweight="bold")

        plt.tight_layout()
        plt.show()

        print("\n--- CLOUD-SYNCED AUDIT LEDGER ---")
        display(pd.DataFrame(ledger).tail(4))

# UI Buttons
btn_normal = widgets.Button(description="Simulate Normal Stream", button_style="success")
btn_anomaly = widgets.Button(description="Inject Critical Failure", button_style="danger")
output_area = widgets.Output()

btn_normal.on_click(lambda b: run_simulation(inject_anomaly=False))
btn_anomaly.on_click(lambda b: run_simulation(inject_anomaly=True))

display(widgets.HBox([btn_normal, btn_anomaly]))
display(output_area)
run_simulation(inject_anomaly=False)

Output()


✅ SUCCESS! Cloud Sync Complete. Firebase generated ID: -P2Dh0FmGZfSsr5DmLX8
